# freeCAM checkpoint Dask fan-out

This Notebook covers restartable Dask task segments and checkpoint fan-out. Each task owns a 24-rank MPI segment, produces a checkpoint Future, and exits. Persistent Actors now have a dedicated `try_persistent_dask.ipynb` Notebook so the two lifetime models are not mixed.

## 1. Execution model

```text
Jupyter/PBS controller + local Dask Client
           │
           ├── base MPI segment: 24 ranks × 10 steps
           │                    │
           │                    └── immutable checkpoint Future
           │                                  │
           ├──────────────────────────────────┼── control:    5 steps
           ├──────────────────────────────────└── no-kessler: 5 steps
           └── warm-initial: +1 K, 0 steps
                         │ exact edit check
                         └── warm: 5 steps
```

The base process exits after writing its checkpoint. Each segment restores private NumPy arrays and creates a new `MPI.COMM_WORLD`; this is checkpoint/restart fan-out, not operating-system `fork()`. With `execution_mode='allocation'`, all segments run serially inside the same outer `PBS_JOBID` and no nested PBS job is submitted.

## 2. Configure one experiment

Run this cell again to create a fresh timestamp before repeating the experiment. Existing branch directories are deliberately never overwritten.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import freecam
from dask.distributed import Client
from netCDF4 import Dataset
from freecam import DaskExperimentClient

repo = Path('/glade/work/$USER/freeCAM')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/$USER'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = (
    repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
)

stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'freecam/dask_notebook_trials' / f'fanout-{stamp}'
initial_run_dir = experiment_root / 'initial-run'
run_root = experiment_root / 'branches'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
dask_workers = 1 if execution_mode == 'allocation' else 3

print('freecam', freecam.__version__)
print('experiment root:', experiment_root)
print('execution mode:', execution_mode)

## 3. Create the Dask controller

The Dask workers orchestrate 24-rank MPI segments. In `allocation` mode one worker serially launches every segment with `mpiexec` inside the current PBS job. In `pbs` mode three workers may independently submit branch PBS jobs.

In [ ]:
if 'client' in globals():
    client.close()

client = Client(
    processes=False,
    n_workers=dask_workers,
    threads_per_worker=1,
    dashboard_address=None,
)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=run_root,
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)
client

## 4. Submit the common base and the zero-step warm edit

This first stage runs two Dask-managed MPI segments. The base runs 10 steps. `warm-initial` restores that exact state, adds `1 K` to `air_temperature`, runs zero model steps, and writes another checkpoint. In allocation mode neither segment submits another PBS job.

In [ ]:
base_plan = experiments.plan('base').step(10)
base = experiments.submit_base(base_plan)

warm_initial_plan = experiments.plan('warm-initial')
warm_initial_plan.fields.edit('air_temperature', 'add', 1.0)
warm_initial = experiments.submit_plan(base, warm_initial_plan)

initial_summaries = experiments.summaries({
    'base': base,
    'warm-initial': warm_initial,
})
initial_summaries

## 5. Verify the edit before model evolution

Both checkpoints are at model step 10. This comparison therefore tests only `warm_initial_plan.fields.edit('air_temperature', 'add', 1.0)` across all 24 ranks. Every edited array must match NumPy's elementwise `base + 1.0` operation bit for bit. Recomputing `(base + 1.0) - base` performs another floating-point operation, so that diagnostic is required to equal `1 K` only within one spacing of the largest base value.

In [ ]:
def checkpoint_field(summary_map, branch, field, rank=0):
    checkpoint_file = (
        Path(summary_map[branch]['checkpoint_dir'])
        / f'rank-{rank:03d}.npz'
    )
    with np.load(checkpoint_file, allow_pickle=False) as arrays:
        return arrays[field].copy()

base_checkpoint = Path(initial_summaries['base']['checkpoint_dir'])
rank_count = len(tuple(base_checkpoint.glob('rank-*.npz')))
base_temperatures = [
    checkpoint_field(initial_summaries, 'base', 'air_temperature', rank)
    for rank in range(rank_count)
]
warm_initial_temperatures = [
    checkpoint_field(
        initial_summaries, 'warm-initial', 'air_temperature', rank
    )
    for rank in range(rank_count)
]
initial_differences = [
    warm - base
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
]
exact_numpy_edit = all(
    np.array_equal(warm, np.add(base, 1.0))
    for base, warm in zip(base_temperatures, warm_initial_temperatures)
)
maximum_roundoff_from_1K = float(
    max(np.abs(delta - 1.0).max() for delta in initial_differences)
)
roundoff_tolerance = float(
    max(np.spacing(np.abs(base).max()) for base in base_temperatures)
)
difference_within_roundoff = (
    maximum_roundoff_from_1K <= roundoff_tolerance
)

assert exact_numpy_edit
assert difference_within_roundoff
sample_count = sum(delta.size for delta in initial_differences)
{
    'ranks': rank_count,
    'rank_local_shape': initial_differences[0].shape,
    'difference_min': float(min(delta.min() for delta in initial_differences)),
    'difference_max': float(max(delta.max() for delta in initial_differences)),
    'difference_mean': float(
        sum(delta.sum() for delta in initial_differences) / sample_count
    ),
    'exact_numpy_add_1K': exact_numpy_edit,
    'maximum_roundoff_from_1K': maximum_roundoff_from_1K,
    'roundoff_tolerance': roundoff_tolerance,
    'difference_within_roundoff': difference_within_roundoff,
}

## 6. Continue the three five-step experiments

`control` and `no-kessler` continue directly from the base checkpoint. `warm` continues from the already verified `warm-initial` checkpoint without applying a second edit. This stage creates three additional Dask tasks; allocation mode executes their full-node MPI segments serially in the same PBS job.

In [ ]:
control_plan = experiments.plan('control').step(5)
no_kessler_plan = experiments.plan('no-kessler', experimental=True)
no_kessler_plan.physics['kessler'].disable()
no_kessler_plan.step(5)
warm_plan = experiments.plan('warm').step(5)

branches = experiments.fork(
    base,
    (control_plan, no_kessler_plan),
)
branches['warm'] = experiments.submit_plan(warm_initial, warm_plan)

summaries = experiments.summaries(branches)
summaries

## 7. What the summary tells you

All three final branches report `step=15` and 16 history samples (the initial state plus 15 completed steps). `control` and `no-kessler` have `parent_branch='base'`; `warm` has `parent_branch='warm-initial'`. The summary also contains `execution_mode`, each PBS job ID, run/history/checkpoint/log paths, and serialized checkpoint size without downloading the full checkpoint Future. In allocation mode every PBS job ID must be identical.

In [ ]:
[
    {
        'branch': name,
        'step': summary['step'],
        'history_samples': summary['history_samples'],
        'execution_mode': summary['execution_mode'],
        'pbs_job_id': summary['pbs_job_id'],
        'checkpoint_GiB': summary['snapshot_nbytes'] / 1024**3,
        'history_dir': summary['history_dir'],
        'log_path': summary['log_path'],
    }
    for name, summary in summaries.items()
]

## 8. Compare the temperature after five model steps

Every branch checkpoint contains the complete 214-field StatePool for all 24 ranks. Unlike the exact pre-run edit, the warm-control difference after five nonlinear model steps is expected to vary around `1 K`. The deviation from `1 K`, rather than the total warm-control difference, measures the subsequent model response.

In [ ]:
control_temperature = checkpoint_field(
    summaries, 'control', 'air_temperature'
)
warm_temperature = checkpoint_field(
    summaries, 'warm', 'air_temperature'
)
temperature_difference = warm_temperature - control_temperature

{
    'rank': 0,
    'shape': control_temperature.shape,
    'difference_min': float(temperature_difference.min()),
    'difference_max': float(temperature_difference.max()),
    'difference_mean': float(temperature_difference.mean()),
    'maximum_deviation_from_1K': float(
        np.abs(temperature_difference - 1.0).max()
    ),
    'bitwise_identical': bool(np.array_equal(control_temperature, warm_temperature)),
}

## 9. Inspect a global field after every model step

Each branch history directory inherits the 10 base timestamps and adds 5 branch timestamps. These NetCDF files contain the 26 configured global diagnostics. Unlike the final rank-local checkpoint, this gives one global field value at every completed model step.

In [ ]:
def history_statistics(branch, variable):
    summary = summaries[branch]
    history_dir = summary.get(
        'history_dir',
        Path(summary['checkpoint_dir']).parent / 'history',
    )
    files = sorted(Path(history_dir).glob('*.nc'))
    records = []
    for path in files:
        with Dataset(path) as dataset:
            values = np.asarray(dataset[variable][0])
            records.append({
                'step': int(dataset['nsteph'][0]),
                'file': path.name,
                'minimum': float(values.min()),
                'maximum': float(values.max()),
                'mean': float(values.mean()),
            })
    return records

control_rain_by_step = history_statistics('control', 'RAINQM')
no_kessler_rain_by_step = history_statistics('no-kessler', 'RAINQM')

{
    'control_last': control_rain_by_step[-1],
    'no_kessler_last': no_kessler_rain_by_step[-1],
    'control_all_steps': control_rain_by_step,
}

## 10. Run phase and scheme actions

`experiments.plan(...)` builds a serializable action sequence without exposing protocol dataclasses. All operations in one plan execute inside one MPI segment and share one live StatePool without writing an intermediate checkpoint. End a plan and call `submit_plan()` when that boundary must become a Future or fork point. Scheme actions use the same DeviceRegistry/CCPP-standard-name connection as the interactive model; Dask does not carry a second kernel implementation. `experimental=True` is required for standalone phase/scheme calls because they do not advance the model clock or automatically run prerequisite calculations.

In [ ]:
granular_plan = experiments.plan(
    'granular-kessler-then-map',
    experimental=True,
)
granular_plan.physics.scheme('kessler', group='before').run()
granular_plan.observe(
    'potential_temperature',
    'large_scale_precipitation_rate',
)
granular_plan.phases['dynamics_to_physics'].run()
granular_plan.observe('air_temperature')

granular = experiments.submit_plan(base, granular_plan)
granular_summary = experiments.summaries({
    'granular': granular,
})['granular']
rank0_potential_temperature = client.gather(
    experiments.field(granular, 'potential_temperature', rank=0)
)

{
    'step': granular_summary['step'],
    'action_trace': granular_summary['action_trace'],
    'rank0_field_shape': rank0_potential_temperature.shape,
    'rank0_field_mean': float(rank0_potential_temperature.mean()),
}

### Make one scheme boundary a separate Future

This form builds a one-operation plan, starts a new 24-rank MPI segment, restores `base`, runs exactly one scheme, and writes a checkpoint. A later `submit_plan()` or `fork()` can use `kessler_only` as its parent.

In [ ]:
kessler_plan = experiments.plan(
    'single-kessler-action', experimental=True
)
kessler_plan.physics.scheme('kessler', group='before').run()
kessler_only = experiments.submit_plan(base, kessler_plan)
experiments.summaries({'kessler': kessler_only})['kessler']

## Dynamic variables and physics plugins in Dask plans

The plan builder exposes the same `fields.create(...)`, `physics.install(...)`, and `physics.scheme(...).run()` vocabulary as a live model. It compiles these calls into the JSON action protocol internally, so users do not construct protocol dataclasses. Inside one plan all operations reuse the same live StatePool; a new Dask Future/checkpoint boundary is created only when `submit_plan()` returns. Source and prebuilt plugin paths must be visible from every PBS worker.

In [ ]:
dynamic_plan_template = experiments.plan(
    'dynamic-extension', experimental=True
)
dynamic_plan_template.fields.create(
    'experiment_tracer',
    dims=('column', 'level'),
    units='kg kg-1',
    initial=0.0,
)
dynamic_plan_template.physics.install(
    repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=repo,
    after='kessler',
    variables=(
        {
            'name': 'runtime_plugin_temperature',
            'standard_name': 'runtime_plugin_temperature',
            'dtype': 'float64',
            'dimensions': ('nphys_local', 'pver'),
            'units': 'K',
        },
        {
            'name': 'runtime_plugin_temperature_increment',
            'standard_name': 'runtime_plugin_temperature_increment',
            'dtype': 'float64',
            'dimensions': (),
            'units': 'K',
            'intent': 'in',
        },
    ),
    inputs={
        'runtime_plugin_temperature': 240.0,
        'runtime_plugin_temperature_increment': 1.5,
    },
)
dynamic_plan_template.physics.scheme(
    'runtime_temperature_offset', group='before'
).run()
dynamic_plan_template.observe('runtime_plugin_temperature')

# dynamic_future = experiments.submit_plan(base, dynamic_plan_template)
dynamic_plan_template.as_dict()

In [ ]:
client.close()
print('Dask client closed; PBS results remain under', experiment_root)

In [ ]:
import os
import socket
import sys

print("host:", socket.gethostname())
print("python:", sys.executable)
print("PBS_JOBID:", os.environ.get("PBS_JOBID"))
print("PBS_NODEFILE:", os.environ.get("PBS_NODEFILE"))